In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/02/27 10:46:29 WARN Utils: Your hostname, codespaces-7a1601, resolves to a loopback address: 127.0.0.1; using 10.0.3.169 instead (on interface eth0)
26/02/27 10:46:29 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/02/27 10:46:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
spark.version

'4.1.1'

In [5]:
!wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet

--2026-02-27 10:55:09--  https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.38.83, 18.239.38.163, 18.239.38.181, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.38.83|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 71134255 (68M) [binary/octet-stream]
Saving to: ‘yellow_tripdata_2025-11.parquet’

yellow_tripdata_202 100%[===================>]  67.84M   286MB/s    in 0.2s    

2026-02-27 10:55:09 (286 MB/s) - ‘yellow_tripdata_2025-11.parquet’ saved [71134255/71134255]



In [26]:
df = spark.read \
    .option("header", "true") \
    .parquet('yellow_tripdata_2025-11.parquet')

In [27]:
df.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+
|       7| 2025-11-01 00:13:25|  2025-11-01 00:13:25|              1|         1.68|         1|                 N|          43|    

In [28]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



In [9]:
df = df.repartition(4)

In [10]:
df.write.parquet('yellow/2025/11/')

In [11]:
import os

path = "yellow/2025/11"

for f in os.listdir(path):
    if f.endswith(".parquet"):
        size = os.path.getsize(os.path.join(path, f))
        print(f, round(size / (1024*1024), 2), "MB")

part-00003-b4620686-153f-4e00-86c7-864f858be3ce-c000.snappy.parquet 25.34 MB
part-00001-b4620686-153f-4e00-86c7-864f858be3ce-c000.snappy.parquet 25.32 MB
part-00002-b4620686-153f-4e00-86c7-864f858be3ce-c000.snappy.parquet 25.32 MB
part-00000-b4620686-153f-4e00-86c7-864f858be3ce-c000.snappy.parquet 25.34 MB


In [17]:
import pyspark.sql.functions as F

date_filter = "2025-11-15"

trips_15_nov = df.filter(F.to_date(F.col("tpep_pickup_datetime")) == date_filter)

count_trips = trips_15_nov.count()

print(f"Number of taxi trips on {date_filter}: {count_trips}")

Number of taxi trips on 2025-11-15: 162604


In [18]:
from pyspark.sql.functions import col, unix_timestamp, max as spark_max


df_with_duration = df.withColumn(
    "duration_hours",
    (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 3600
)

max_duration = df_with_duration.select(spark_max("duration_hours")).collect()[0][0]

print(f"Longest trip duration: {max_duration:.2f} hours")

[Stage 31:=============================>                            (2 + 2) / 4]

Longest trip duration: 90.65 hours


In [19]:
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2026-02-27 11:46:31--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 18.239.38.163, 18.239.38.83, 18.239.38.181, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|18.239.38.163|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0s      

2026-02-27 11:46:31 (472 MB/s) - ‘taxi_zone_lookup.csv’ saved [12331/12331]



In [29]:
zones_df = spark.read \
    .option("header", "true") \
    .csv('taxi_zone_lookup.csv')

In [30]:
zones_df.show()

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
|         6|Staten Island|Arrochar/Fort Wad...|   Boro Zone|
|         7|       Queens|             Astoria|   Boro Zone|
|         8|       Queens|        Astoria Park|   Boro Zone|
|         9|       Queens|          Auburndale|   Boro Zone|
|        10|       Queens|        Baisley Park|   Boro Zone|
|        11|     Brooklyn|          Bath Beach|   Boro Zone|
|        12|    Manhattan|        Battery Park| Yellow Zone|
|        13|    Manhattan|   Battery Park City| Yellow Zone|
|        14|     Brookly

In [33]:
from pyspark.sql.functions import col, count


pickup_counts = df.groupBy("PULocationID").agg(count("*").alias("pickup_count"))

pickup_with_names = pickup_counts.join(
    zones_df,
    pickup_counts.PULocationID == zones_df.LocationID,
    "left"
).select("Zone", "pickup_count")

least_frequent = pickup_with_names.orderBy("pickup_count").first()

print(f"Least frequent pickup zone: {least_frequent['Zone']} with {least_frequent['pickup_count']} pickups")

[Stage 44:>                                                         (0 + 2) / 2]

Least frequent pickup zone: Governor's Island/Ellis Island/Liberty Island with 1 pickups
